# 🌍 Worlds and Profiles — where the swapping actually happens

Since `03_protocols_and_ports.ipynb` this course has been making one promise: *build
against ports, and reality can be swapped in underneath without touching the logic.* You
have since met every real organ — the contracts (06), the signing adapter (07), the
bouncer (08), the hands (09), the brains (10). This chapter shows the ONE file where the
swapping is actually performed — `e2e/src/e2e/skeleton/worlds.py`, the **composition
root** — and the environment contract, `SKELETON_PROFILE`, that picks which reality a
test run gets.

**What you'll be able to do after this notebook**

- Point at the composition root and explain why exactly *one* place may know which adapter fills which port
- Read `SKELETON_PROFILE = mock | chain | chain+net | full` as a contract between you, pytest, and CI
- Walk the canonical lifecycle by hand through `MockWorld` — the very code path CI walks on every commit
- Perform THE SWAP live: the identical script on a real Anvil chain — real ERC-721, real payment, real revocation event
- Explain what `just up` / `just down` start and stop, and how fixture finalizers police zombie processes and zombie router config

**You need:** notebooks 00–10 (03 and 05 do the heavy lifting; 06 and 07 are what make
§5 meaningful). Infrastructure: none for §§1–4 and 6–8; §5 wants `anvil` on PATH plus
forge-built artifacts (`cd contracts && forge build`) and politely skips without them.
**Estimated time:** 50–60 minutes.

## 1. One place knows what's real (the composition root)

In 05 you wired the skeleton by hand: build a `FakeChain`, build a `FakeNet`, hand both
to `StubController`. Convenient in a notebook — but imagine the controller instead did
`from e2e.skeleton.fakes import FakeChain` *inside its own file*. The day you want the
real chain, you edit the controller. The day you want the real router, you edit it
again. Every swap becomes surgery on the very logic the swap was supposed to leave
alone, and 03's hexagon collapses.

The cure has a name: the **composition root** — the single place in a program where
"the controller needs an `EntitlementReader`" meets "today, that reader is a
`FakeChain`". Adapters are chosen there and *only* there; everything else just declares
which port it needs and stays ignorant. In this repo the composition root for the
lifecycle is `worlds.py`, and its docstring reads like a stage direction: one script,
three stagings.

In [ ]:
import inspect
import sys
from datetime import datetime, timezone
from pathlib import Path

import a2a_interfaces.fixtures as fx
from a2a_interfaces import EntitlementReader, ErrorCode, SessionState

import e2e.skeleton.worlds as worlds
from e2e.skeleton.fakes import InsufficientFunds, OfferAlreadyUsed, WrongConsumer
from e2e.skeleton.stub_controller import Denied
from e2e.skeleton.worlds import MALLORY, STORY_TIME, ChainNetWorld, ChainWorld, MockWorld

ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / ".git").exists())


def hhmm(ts: int) -> str:
    """Unix seconds -> the story's HH:MM (UTC), for readable narration."""
    return datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%H:%M")


print("\n\n".join(inspect.getdoc(worlds).split("\n\n")[:2]))
assert STORY_TIME == fx.WINDOW.start - 1680          # both worlds open at 13:32 (ADR-004)
print("\nthree stagings:", MockWorld.__name__, "·", ChainWorld.__name__, "·", ChainNetWorld.__name__)

Now the detail that proves the design is deliberate. `worlds.py` *contains*
`ChainWorld`, which needs `web3`, and `ChainNetWorld`, which needs `pygnmi` — yet the
`import` lines for both live **inside method bodies**, not at the top of the file
(you can see them in §3's sources). So importing the composition root costs nothing it
doesn't need yet: CI's mock profile never touches a chain or router library. Check what
our import actually dragged in — `sys.modules` is Python's ledger of every module loaded
so far:

In [ ]:
print("web3   loaded?", "web3" in sys.modules, "   ← ChainWorld's chain library — not yet")
print("pygnmi loaded?", "pygnmi" in sys.modules, "   ← ChainNetWorld's router library — not yet")
print("controller loaded?", "controller" in sys.modules, "← the REAL predicate came along (05 §7)")
assert "web3" not in sys.modules and "pygnmi" not in sys.modules
assert "controller" in sys.modules
print("✓ importing the composition root pulls in only what the mock staging needs")

**✏️ Your turn 1.1 — predict the import bill**

The repo has five Python packages: `a2a_interfaces`, `e2e`, `netctl`, `controller`,
`chainmcp`. Write your prediction as a comment — which of the five are ALREADY in
`sys.modules` after importing `worlds`, and which are not? — then run the cell and
un-comment the self-check. Success: exactly one package is missing, and it is the one
that holds private keys.

In [ ]:
# TODO: your prediction first, as a comment — which package is NOT loaded yet, and why?
# my_prediction = "..."

repo_pkgs = ("a2a_interfaces", "e2e", "netctl", "controller", "chainmcp")
loaded = [p for p in repo_pkgs if p in sys.modules]
missing = [p for p in repo_pkgs if p not in sys.modules]
print("loaded :", loaded)
print("missing:", missing)

# un-comment once you've predicted:
# assert missing == ["chainmcp"]

<details><summary>✅ Solution 1.1 — peek only after trying</summary>

```python
assert missing == ["chainmcp"]
```

`chainmcp` (the key holder, rule 2) is deferred into `ChainWorld.__init__`; `netctl`
*is* loaded because `FakeNet` is a plain alias for `netctl.mock.MockProvisioner`
(05 §13), and `controller` because the stub employs the real predicate — exactly one
bouncer exists in the codebase.
</details>

## 2. `SKELETON_PROFILE` — an environment contract

If the composition root can stage the play three ways, something must pick the staging —
without editing any test. That something is an **environment variable** (a named string
the shell hands a process at launch; you met the `LLM_*` trio in 10). The contract:

| profile | chain port | network port | judgment slots | who runs it |
|---|---|---|---|---|
| `mock` *(default)* | `FakeChain` | `FakeNet` (= `MockProvisioner`) | scripted | CI, every commit, forever |
| `chain` | live Anvil + real contracts, via `ChainClient`s | `FakeNet` | scripted, but the signature is real | you, locally, in seconds |
| `chain+net` | live Anvil | `GnmiProvisioner` → real SR Linux | scripted | you, with the lab up |
| `full` | live Anvil | mock or lab | real LangGraph agents + an LLM | you, with an endpoint |

Note where the selection does **not** live: not in `worlds.py`, and never inside a
test. It lives in `e2e/tests/conftest.py` — a file pytest loads automatically to share
setup. Read its opening: the docstring states the contract, one line reads the
environment, and the default is the reality that needs nothing.

In [ ]:
conftest_path = ROOT / "e2e" / "tests" / "conftest.py"
conftest_src = conftest_path.read_text()

print(conftest_src[: conftest_src.index("@pytest.fixture")].strip())
assert 'os.environ.get("SKELETON_PROFILE", "mock")' in conftest_src
print("\n✓ default = mock — CI needs no anvil, no router, no LLM, ever")

The docstring's last sentence is the honesty policy: ask for `chain` without anvil and
you get **an error, not a skip** — a requested reality silently degrading to mock would
be a green checkmark that lies.

Now the delivery mechanism. A pytest **fixture** is a named setup function; a test that
lists `world` as a parameter receives whatever the fixture `yield`s, and the code
*after* the `yield` runs as cleanup — the **finalizer** — even when the test fails.
Read the `world` fixture: mock short-circuits in two lines; the chain path snapshots the
chain before each test and reverts after (`evm_snapshot` / `evm_revert` — a save-game
and its reload, so every test starts from the same freshly-seeded chain).

In [ ]:
print(conftest_src[conftest_src.index("@pytest.fixture()") :])
assert "evm_snapshot" in conftest_src and "chain_world.close()" in conftest_src

**✏️ Your turn 2.1 — find where a wrong profile dies**

conftest refuses in exactly two places rather than degrade. Grep the source for
`RuntimeError`, then read each hit in its function above. Success: you can say which of
the two would fire for `SKELETON_PROFILE=banana`, and which for `SKELETON_PROFILE=chain`
on a machine without Foundry.

In [ ]:
refusals = [ln.strip() for ln in conftest_src.splitlines() if "RuntimeError" in ln]
print(*refusals, sep="\n")

# TODO: scroll back to the two conftest printouts above (the docstring, then the
#       `world` fixture) — which refusal answers `banana`, which answers
#       `chain`-without-anvil?
# un-comment once you know:
# assert len(refusals) == 2
# assert any("unknown SKELETON_PROFILE" in ln for ln in refusals)

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
assert len(refusals) == 2
assert any("unknown SKELETON_PROFILE" in ln for ln in refusals)
```

`banana` dies in the `world` fixture (`unknown SKELETON_PROFILE=...`); `chain` without
anvil/artifacts dies in `_anvil_session` — both are loud, because a profile you asked
for must never quietly become a different one.
</details>

## 3. `worlds.py`, binding by binding

Time to read the bindings themselves. `MockWorld.__init__` is 05's final wiring cell,
boxed: every attribute is one port (or judgment slot) filled with a cardboard prop.
Notice there is no cleverness here — a composition root is *supposed* to be boring
assignments, because this is the code you read when you ask "what is real today?".

In [ ]:
print(inspect.getsource(MockWorld.__init__))
src = inspect.getsource(MockWorld.__init__)
assert "FakeChain" in src and "FakeNet" in src and "StubController" in src
print("→ every port filled with a fake; both judgment slots scripted")

`ChainWorld.__init__` fills the *same slots* with real things: one `ChainClient` per
key holder (07's custody rule — identity *is* the client), Bell's judgment slot upgraded
to `_SigningProvider` so the quote carries a **real** EIP-712 signature, and a
`threading.Event` — a flag one thread raises with `.set()` while another sleeps on it
in `.wait()` (01 taught the `Thread` half; the flag is new here) — to bridge the
chain's asynchronous `Revoked` event back into a synchronous script. Two honesty notes hide in the comments: the controller reads through
Carol's client — a flagged v1 shortcut, since a controller holding *any* signing power
strains rule 2's spirit — and `net` is still `FakeNet`: this staging swaps the chain
organ only.

In [ ]:
print(inspect.getsource(ChainWorld.__init__))
src = inspect.getsource(ChainWorld.__init__)
assert "ANVIL_KEYS" in src and "watch_revoked" in src
assert "from chainmcp.testing import ANVIL_KEYS" in src   # the deferred import, in the flesh

And the third staging is barely a class at all. `ChainWorld.__init__` builds its network
leg by calling `self._make_net()` — a **template method** hook: the parent defines the
skeleton of the algorithm and leaves one step for a subclass to override.
`ChainNetWorld(ChainWorld)` overrides that hook to return a `GnmiProvisioner` aimed at
the real router (09), plus the two inspection helpers that now read *off the router*
instead of out of a dict. The lifecycle methods — quote, fulfill, advance, revoke — are
inherited untouched. The play's lines never change; only the stage crew does.

**✏️ Your turn 3.1 — predict what `ChainNetWorld` redefines**

Before running: write down which method names you expect to find in `ChainNetWorld`'s
own body (hint: the previous paragraph names them). Then run and un-comment the check.
Success: `fulfill` is *inherited*, and the list of self-defined names is exactly the
network-shaped ones.

In [ ]:
# TODO: your prediction — which methods does ChainNetWorld define itself?
# my_prediction = ["...", "..."]

own = sorted(n for n in vars(ChainNetWorld) if not n.startswith("__"))
inherited = sorted(
    n for n in vars(ChainWorld) if not n.startswith("__") and n not in vars(ChainNetWorld)
)
print("defines itself:", own)
print("inherits      :", inherited)

# un-comment once you've predicted:
# assert "fulfill" not in own and "_make_net" in own

<details><summary>✅ Solution 3.1 — peek only after trying</summary>

```python
assert own == ["_make_net", "close", "provisioned", "torn_down"]
```

Four names: the network hook, the two reality-readers, and `close` (the zombie sweep,
§6) — everything about *commerce* is inherited unchanged, which is the whole argument
for the template-method shape.
</details>

## 4. The mock world walks the whole play

Build the default staging and check the stage: curtain at 13:32, Ada holds 50 TOK, Bell
none, and the next ticket number is 7. Keep one thing in mind for the whole section —
**these are the exact steps `e2e/tests/test_lifecycle.py` performs in CI on every
commit**. The notebook and the test are twins; we will hold them side by side in a
moment.

In [ ]:
w = MockWorld()

print("reader      :", type(w.reader).__name__, "   net:", type(w.net).__name__)
print("chain time  :", hhmm(w.reader.chain_time()), "            ← before Ada's 14:00 window")
print("Ada balance :", w.balance_of(fx.ADA) // 10**18, "TOK")
print("Bell balance:", w.balance_of(fx.BELL) // 10**18, "TOK")
assert w.balance_of(fx.ADA) == 50 * 10**18 and w.balance_of(fx.BELL) == 0

Scenes 1–3 are off-chain messages: Bell's slot quotes, Ada's slot accepts (both canned —
rule 1's two judgment slots, hardcoded since 05 §6). Then, before Ada pays, let the
story's freeloader go first: Mallory holds gas money but zero TOK and grabs the open
offer anyway. Watch two things — the exception class, and that the rejected `fulfill`
left **no trace** (05 §5's atomicity, now asserted through the world surface). Remember
this scene; §5 replays it against a real ERC-20.

In [ ]:
signed = w.provider.quote(None)                          # Bell's judgment slot (canned)
decision = w.consumer.decide(fx.BANDWIDTH_NEED, signed)  # Ada's judgment slot (canned)
assert decision.accept
print(f'13:32  Bell quotes 50 Mbps / 10 TOK; Ada: {{"accept": {decision.accept}}}')

print("Mallory's TOK:", w.balance_of(MALLORY), " ← gas money yes, TOK none")
try:
    w.fulfill(signed, buyer=MALLORY)
    raise AssertionError("should have raised")
except InsufficientFunds as e:
    print("✗ InsufficientFunds:", str(e)[:10] + "…", "← the freeloader, refused")
assert not w.salt_consumed(signed) and w.balance_of(fx.BELL) == 0
print("no trace ✓ — the rejected fulfill mutated nothing")

Scene 4, the one on-chain write: Ada redeems. Note that the assertions are **deltas**
(before/after differences), not absolute balances — a habit the twin test explains: on
the chain profile Bell has *already* earned 60 TOK selling tickets #1–#6, so pinning
absolutes would couple the script to stage dressing. Deltas survive every staging.

In [ ]:
ada_before, bell_before = w.balance_of(fx.ADA), w.balance_of(fx.BELL)

eid = w.fulfill(signed, buyer=fx.ADA)

print(f"13:32  fulfill() -> ticket #{eid}   owner: {w.reader.owner_of(eid)[:8]}… (Ada)")
print(f"       Ada  {ada_before // 10**18} -> {w.balance_of(fx.ADA) // 10**18} TOK")
print(f"       Bell {bell_before // 10**18} -> {w.balance_of(fx.BELL) // 10**18} TOK")
assert eid == fx.TICKET_ID and w.reader.owner_of(eid) == fx.ADA
assert w.balance_of(fx.ADA) == ada_before - int(fx.PRICE_10_TOK)
assert w.balance_of(fx.BELL) == bell_before + int(fx.PRICE_10_TOK)
assert w.salt_consumed(signed)

Now the trap every demo of this repo hits exactly once. It is 13:32; Ada's window opens
at 14:00. Owning ticket #7 does not bend time — activate now and the bouncer says *not
yet*. Break it on purpose, so the deny is a friend when you meet it in your own
experiments:

In [ ]:
try:
    w.controller.activate(eid, requester=fx.ADA, nonce=w.controller.challenge(eid))
    raise AssertionError("should have been denied")
except Denied as e:
    print(f"✗ activate at {hhmm(w.reader.chain_time())} ->", e.code.value, " ← window opens at 14:00")
    assert e.code == ErrorCode.E_NOT_STARTED

Scenes 5–8: advance chain time 30 story-minutes to 14:02, then the challenge → activate
dance from 05 §8. The line to appreciate is `w.advance_time(...)`: in this world it nudges
a `FakeClock`; in §5's world the SAME call mines a real block with a warped timestamp.
The script does not know, and never needs to.

In [ ]:
w.advance_time(1800)                                     # 13:32 + 30 min -> 14:02

sid = w.controller.activate(eid, requester=fx.ADA, nonce=w.controller.challenge(eid))
print(f"{hhmm(w.reader.chain_time())}  activate() -> {sid!r}   state: {w.controller.state(sid).value}")
print("       provisioned:", w.provisioned(sid))
assert w.controller.state(sid) == SessionState.ACTIVE
assert w.provisioned(sid) == {"capacity_bps": fx.CAPACITY_50_MBPS, "qos_class": fx.QOS_CLASS}

Scene 9, the time-driven ending: jump to the window's end and `tick()`. The controller
re-reads `chain_time()` before acting (ADR-004 — the wake-up schedules, the chain
decides), tears the session down, and — rule 8 — tearing it down *again* is a success,
not an error. (The other ending, revocation, is your exercise below.)

In [ ]:
view = w.reader.get(eid)
w.advance_time(view.end_time - w.reader.chain_time())    # jump to exactly 16:00
w.controller.tick()

print(f"{hhmm(w.reader.chain_time())}  tick() -> {w.controller.state(sid).value}   torn_down: {w.torn_down(sid)}")
assert w.controller.state(sid) == SessionState.TORN_DOWN and w.torn_down(sid)

w.controller.teardown(sid)                               # once more, on purpose
assert w.torn_down(sid)
print("✓ rule 8: tearing down what's already down is a success")

Now meet the twin. Open `e2e/tests/test_lifecycle.py` beside this notebook — its happy
path is the walk you just did, step for step, and its `world` parameter is §2's fixture.
That is the profile machinery's whole payoff: this one function is the CI play on
cardboard *and* the local play on a real chain, depending on one environment variable.
Try it in a terminal, with narration:

```bash
uv run pytest e2e/tests/test_lifecycle.py -s          # mock — CI's reality, sub-second
```

In [ ]:
lifecycle_src = (ROOT / "e2e" / "tests" / "test_lifecycle.py").read_text()
happy = lifecycle_src[
    lifecycle_src.index("def test_happy_path_lifecycle") : lifecycle_src.index("def test_replayed")
]
print(happy)
for step in ("world.fulfill", "world.advance_time", "world.controller.activate", "world.controller.tick"):
    assert step in happy
print("✓ the very steps you just ran by hand — the notebook and the test are twins")

**✏️ Your turn 4.1 — run the other ending**

The scaffold below re-runs the play to ACTIVE on a *fresh* world. Your job: give it the
event-driven ending — Bell pulls ticket #7 mid-window (one world call; 05 §10 showed the
cardboard mechanism). Success: the state flips to `TORN_DOWN` with **no** `tick()`, and
the ticket reads back `revoked=True`.

In [ ]:
w2 = MockWorld()
eid2 = w2.fulfill(w2.provider.quote(None), buyer=fx.ADA)
w2.advance_time(1800)                                    # -> 14:02, inside the window
sid2 = w2.controller.activate(eid2, requester=fx.ADA, nonce=w2.controller.challenge(eid2))
print("before:", w2.controller.state(sid2).value)

# TODO: Bell revokes ticket #7 — one call on the world, no tick()
# ...

print("after :", w2.controller.state(sid2).value)
# un-comment once done:
# assert w2.controller.state(sid2) == SessionState.TORN_DOWN
# assert w2.torn_down(sid2) and w2.reader.get(eid2).revoked

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
w2.revoke(eid2)
```

In `MockWorld` this rings `FakeChain`'s callbacks synchronously; in `ChainWorld` the
*same call* sends a real transaction and then blocks until a watcher thread has seen the
`Revoked` event — same script line, two realities.
</details>

## 5. THE SWAP, live — same script, real chain

Everything above was cardboard. Now the punchline this course has been building toward:
run the **identical** lifecycle with the chain organ real. We launch a disposable Anvil
frozen at 13:32 (06), deploy the real contracts, and run `seed_chain` — the stage
dressing that funds the cast and has Bell pre-sell tickets #1–#6, so the canonical
purchase literally mints **#7** (the mock's `next_id=7`, made physical).

This section needs `anvil` on PATH and forge-built artifacts. Without them every cell
below prints a skip line instead of failing — the notebook stays honest either way.

In [ ]:
from chainmcp.testing import anvil_available, artifacts_available

CHAIN_OK = anvil_available() and artifacts_available()
if not CHAIN_OK:
    print("skipped: needs anvil + forge artifacts — install Foundry (foundryup), run")
    print("         `cd contracts && forge build`, then rerun from this cell")
else:
    from chainmcp.testing import launch_anvil

    from e2e.skeleton.worlds import seed_chain

    anvil = launch_anvil(timestamp=STORY_TIME)     # throwaway chain, frozen at 13:32
    seed_chain(anvil)                              # fund the cast; Bell pre-sells #1–#6
    print("anvil:", anvil.rpc_url, "  chain time:", hhmm(STORY_TIME))
    print("✓ stage dressed — Ada funded, six tickets pre-sold, next mint is #7")

Build the world. Compare this cell to §4's `w = MockWorld()`: the only difference the
script can see is *which class we constructed*. Print the reader binding side by side —
a dict-backed fake next to a live RPC client — and cash 03's punchline: **both satisfy
`EntitlementReader`**, which is precisely why nothing downstream needs to change.

In [ ]:
if not CHAIN_OK:
    print("skipped: needs anvil + forge artifacts (see the §5 guard cell)")
else:
    cw = ChainWorld(anvil)

    print(f"mock world reader : {type(w.reader).__name__:12} ← a dict in this process")
    print(f"chain world reader: {type(cw.reader).__name__:12} ← a live JSON-RPC client")
    assert isinstance(w.reader, EntitlementReader) and isinstance(cw.reader, EntitlementReader)
    print("both satisfy EntitlementReader ✓   (03's punchline, cashed)")

    print("Ada's TOK on the REAL chain:", cw.balance_of(fx.ADA) // 10**18, "TOK ← seeded")
    assert cw.balance_of(fx.ADA) == 50 * 10**18

Same play, scene by scene. First the quote — Bell's judgment is still scripted, but the
signature is now 65 **real** bytes from Bell's key (signing lives only in `chainmcp`,
rule 2; you built this in 07). Then Mallory again, word for word from §4 — except this
time the refusal is a genuine ERC-20 revert from the token contract, arriving in the
*same exception class* the fake raised. Look at the error's text: the chain's spelling
shows through.

In [ ]:
if not CHAIN_OK:
    print("skipped: needs anvil + forge artifacts (see the §5 guard cell)")
else:
    signed_c = cw.provider.quote(None)
    print("signature:", signed_c.signature[:22] + "…", "← real bytes, recoverable to Bell")
    assert signed_c.signature != "0x" + "ab" * 65        # not 05's placeholder

    try:
        cw.fulfill(signed_c, buyer=MALLORY)
        raise AssertionError("should have raised")
    except InsufficientFunds as e:
        print("✗ InsufficientFunds:", e, " ← the ERC-20 revert, wearing the fake's name")
    assert not cw.salt_consumed(signed_c)
    print("no trace on the real chain either ✓ (a revert rolls back everything)")

How did an on-chain revert become the *skeleton's* exception? At the fulfill edge,
`ChainWorld` keeps a translation table: `ChainRevert` names → the fake's exception
classes. Three names match one-for-one (that parity was built deliberately in M1.3);
the two ERC-20 spellings both mean "broke". The re-raise uses `raise exc(...) from err`
— **exception chaining**, which keeps the original revert attached as the cause instead
of discarding evidence. One script, two realities, identical failure surface: that is
rule 7 *in action*, not just in a contract test.

In [ ]:
for name, exc in ChainWorld._ERRORS.items():
    print(f"{name:26} -> {exc.__name__}")
print()
print(inspect.getsource(ChainWorld._fulfill_as))
assert ChainWorld._ERRORS["ERC20InsufficientBalance"] is InsufficientFunds

Scene 4 again — and this time `fulfill` is a real transaction: an EIP-712 signature
verified in Solidity, 10 real TOK pulled through an allowance, and a real **ERC-721**
minted. Watch Bell's balance: 60 TOK *before* Ada pays, because the six pre-sales were
real sales — exactly why the assertions are deltas.

In [ ]:
if not CHAIN_OK:
    print("skipped: needs anvil + forge artifacts (see the §5 guard cell)")
else:
    ada_before, bell_before = cw.balance_of(fx.ADA), cw.balance_of(fx.BELL)

    eid_c = cw.fulfill(signed_c, buyer=fx.ADA)

    print(f"fulfill() -> ticket #{eid_c}  ← a REAL ERC-721, owner {cw.reader.owner_of(eid_c)[:8]}… (Ada)")
    print(f"Ada  {ada_before // 10**18} -> {cw.balance_of(fx.ADA) // 10**18} TOK")
    print(f"Bell {bell_before // 10**18} -> {cw.balance_of(fx.BELL) // 10**18} TOK  ← the 60 from six pre-sales")
    assert eid_c == fx.TICKET_ID and cw.reader.owner_of(eid_c) == fx.ADA
    assert cw.balance_of(fx.ADA) == ada_before - int(fx.PRICE_10_TOK)
    assert cw.balance_of(fx.BELL) == bell_before + int(fx.PRICE_10_TOK)
    assert cw.salt_consumed(signed_c)                    # read back from the real contract

Activation, unchanged: `advance_time` now warps the real chain (`evm_increaseTime` plus
a mined block — chain time is still the only clock, ADR-004), and `chain_time()` reads a
real block's timestamp. One deliberate anticlimax to notice: `cw.net` is **still
`FakeNet`**. The `chain` staging swaps exactly one organ — chain real, network cardboard
— because a swap you can't isolate is a swap you can't debug. The router's turn is the
`chain+net` profile (§6).

In [ ]:
if not CHAIN_OK:
    print("skipped: needs anvil + forge artifacts (see the §5 guard cell)")
else:
    cw.advance_time(1800)                                # evm_increaseTime + one mined block
    print("chain time:", hhmm(cw.reader.chain_time()), "← a real block's timestamp")

    sid_c = cw.controller.activate(eid_c, requester=fx.ADA, nonce=cw.controller.challenge(eid_c))
    print("state:", cw.controller.state(sid_c).value, "  provisioned:", cw.provisioned(sid_c))
    assert cw.controller.state(sid_c) == SessionState.ACTIVE
    assert type(cw.net).__name__ == "MockProvisioner"
    print("✓ chain real, network still cardboard — one organ per swap")

And the ending §4 left to the exercise: revocation — now a **real on-chain event**.
`cw.revoke(...)` has Bell send a real transaction, then blocks on a `threading.Event`
until the watcher thread (polling the chain every 0.2 s) has delivered the event; the
controller's own callback registered first, so by the time the wait returns, the session
is already down. The full showpiece — with the *real* controller instead of the stub —
is `12_grand_finale.ipynb`'s finale.

In [ ]:
if not CHAIN_OK:
    print("skipped: needs anvil + forge artifacts (see the §5 guard cell)")
else:
    cw.revoke(eid_c)         # real tx, then BLOCKS until the watcher saw the Revoked event

    print("after revoke:", cw.controller.state(sid_c).value, "  torn_down:", cw.torn_down(sid_c))
    print("ticket #7 on chain: revoked =", cw.reader.get(eid_c).revoked, " ← still readable, never burned")
    assert cw.controller.state(sid_c) == SessionState.TORN_DOWN and cw.torn_down(sid_c)
    assert cw.reader.get(eid_c).revoked

One difference from pytest worth naming before you experiment: conftest wraps every test
in `evm_snapshot` / `evm_revert`, so each starts from the pristine seeded chain. A
notebook keeps **one** anvil across cells, so state accumulates — the canonical salt is
now spent for good on this chain, and rerunning the purchase cell above would raise
`OfferAlreadyUsed`. Fresh offers need fresh salts; keep that in mind for the exercise.

**✏️ Your turn 5.1 — a real deny path, predicted**

`cw.quote_targeted(fx.ADA)` has Bell *re-sign* the offer with the consumer bound to Ada
(on a real chain you cannot retarget after signing — the contract would answer
`BadSignature`; that is why the method re-signs instead of copying). Predict which of
the four exception classes fires when **Bell** tries to redeem Ada's offer — then run
and un-comment the check. Success: the same class 05 §5 taught you, translated from a
real revert.

In [ ]:
if not CHAIN_OK:
    print("skipped: needs anvil + forge artifacts (see the §5 guard cell)")
else:
    targeted = cw.quote_targeted(fx.ADA)   # re-SIGNED, consumer = Ada
    # TODO: predict — OfferExpired, WrongConsumer, OfferAlreadyUsed, or InsufficientFunds?
    # my_prediction = "..."

    caught = None
    try:
        cw.fulfill(targeted, buyer=fx.BELL)
    except Exception as e:
        caught = e
        print("->", type(e).__name__, ":", e)

    # un-comment once you've predicted:
    # assert isinstance(caught, WrongConsumer)

<details><summary>✅ Solution 5.1 — peek only after trying</summary>

```python
assert isinstance(caught, WrongConsumer)
```

The consumer binding is first in the contract's revert order (expired → consumer →
replay → funds), so Bell is refused before anything else is examined — and the revert
name `WrongConsumer` travels through `_ERRORS` into the very class the fake raises.
One subtlety: on-chain the replay ledger is keyed by the offer **digest**, and
re-signing with `consumer = Ada` produced a *fresh* digest — never spent, even though
the salt is the canonical one. That is why `test_lifecycle.py`'s twin of this exercise
can still fulfill the same targeted offer with *Ada* right after Bell's refusal
("she still can"). Only the fake's ledger keys by salt.
</details>

## 6. What `chain+net` adds — and what `full` adds

`chain+net` changes one binding: `_make_net` returns a `GnmiProvisioner` aimed at the
lab's `srl1` (09's hands), found via `lab_ipv4()`. Read the hook — and note the refusal
when the lab is down. Same honesty policy as conftest: you asked for the router, so no
router is an **error**, never a silent mock.

In [ ]:
print(inspect.getsource(ChainNetWorld._make_net))
assert "lab_ipv4" in inspect.getsource(ChainNetWorld._make_net)

try:
    from netctl.testing import lab_ipv4
    ip = lab_ipv4()
except FileNotFoundError:                    # no docker binary on this machine
    ip = None
print("lab_ipv4() ->", ip, " ← None = no live lab; ChainNetWorld would refuse to build")

The deeper change is *where truth lives*. In the mock and chain stagings,
`provisioned(sid)` reads a recording dict; in `chain+net` it queries the actual policer
**off the router** via gNMI, and `torn_down` means "no policer left on the box". The
assertion is against reality, not against a note we wrote to ourselves:

In [ ]:
print(inspect.getsource(ChainNetWorld.provisioned))
assert "peak-rate-kbps" in inspect.getsource(ChainNetWorld.provisioned)
print("→ same method name, but the answer now comes from srl1 itself")

**✏️ Your turn 6.1 — read the zombie sweep**

`ChainNetWorld.close()` must leave the router clean **even when the test that used it
failed** — a leftover policer would throttle every later measurement into a lie. Read
the source below and find: (a) the string prefix that marks the policers this project
owns, and (b) the Python keyword that guarantees the gNMI client closes even if the
sweep itself blows up (01 met it). Success: both asserts pass.

In [ ]:
close_src = inspect.getsource(ChainNetWorld.close)
print(close_src)

# TODO: (a) which prefix marks OUR policers?  (b) which keyword guarantees the cleanup?
# un-comment once found:
# assert '"a2a-"' in close_src        # (a) the naming scheme
# assert "finally" in close_src       # (b) cleanup that runs no matter what

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
assert '"a2a-"' in close_src
assert "finally" in close_src
```

Every policer this system creates is named `a2a-<session>`, so the sweep can delete its
own leftovers and nobody else's; `try/finally` makes the client close and the
`super().close()` chain run even when the sweep fails — the fixture finalizer calls this
`close`, so a *failed* test still polices its zombies.
</details>

And `full`? It has **no world class at all**. The fourth profile lives in
`e2e/tests/test_full_agent_driven.py`, where the real LangGraph consumer and provider
graphs (10) replace the scripted judgment slots, with a live LLM endpoint (the `LLM_*`
environment trio) or the test's fake LLM standing in. Worlds swap adapters below the
judgment line; `full` swaps the judgment itself — the last cardboard gone.

Terminal recipes, in escalation order:

```bash
# skeleton v1 — the SAME tests, real chain (seconds):
SKELETON_PROFILE=chain uv run pytest e2e/ -s

# skeleton v2 — bring the router lab up first (~1 min, ~2 GB):
sudo containerlab deploy -t netlab/topology.clab.yml
SKELETON_PROFILE=chain+net uv run pytest e2e/tests/test_lifecycle.py -s

# the full profile — real agent graphs (LLM_* env optional; see llmserve/README.md):
uv run pytest e2e/tests/test_full_agent_driven.py -s
```

## 7. `bringup.py` — the one-command orchestration

Pytest builds worlds *in-process* and tears them down per test. But the operator console
(and your own manual poking) wants a **standing stack**: an Anvil, deployed contracts,
and the controller HTTP server (the thin `e2e/src/e2e/controller_server.py`, which wires
a `ChainReader` + `MockProvisioner` into 08's FastAPI app and serves it). That is
`just up` — implemented by `e2e/src/e2e/bringup.py` as three plain background processes,
no containers.

A heads-up before the import: `bringup.py` runs code at module level — it resolves the
repo root and **registers a SIGTERM handler in the importing process**. SIGTERM is a **signal**:
the operating system's polite "please stop" message to a process — what the `kill`
command sends by default (Ctrl-C's cousin) — and the handler is the function that
runs when it arrives. So a SIGTERM to this kernel would now run `down()`.
Importing a module executes it; this one makes that
lesson impossible to miss.

In [ ]:
import e2e.bringup as bringup    # side effects: resolves repo root, registers a SIGTERM handler

print("manifest path :", bringup.MANIFEST.relative_to(ROOT))
print("fixed ports   : anvil", bringup.ANVIL_PORT, " · controller", bringup.CONTROLLER_PORT)
print("story time    :", hhmm(bringup.STORY_TIME), "← the same 13:32 as every world")
print("stack running?", bringup.MANIFEST.exists(),
      "  anvil port open?", bringup._port_open(8545), " ← _private: peeking to learn")
assert bringup.STORY_TIME == STORY_TIME

Read `up()` as three beats of *start, then wait*: Anvil (wait for the port), the one-shot
`forge script` deploy, the controller under `uv run` (wait for a healthy HTTP answer).
Two details carry the design. `start_new_session=True` makes each child the leader of its
own **process group** — a family of processes that can be signalled together — so `down`
can later kill *grandchildren* (uvicorn is a child of `uv run`, not of `up`). And the
last act writes a **manifest** — PIDs and URLs to `e2e/runs/current.json` — the only
link that lets a *different shell*, hours later, find and stop what this one started.

In [ ]:
print(inspect.getsource(bringup.up))
up_src = inspect.getsource(bringup.up)
assert "start_new_session=True" in up_src and "manifest.save()" in up_src

The waiting half: `_wait` polls any zero-argument callable until it answers `True` (the
broad `except` is acceptable *only* because retrying is the handling); `_port_open` is a
cheap TCP knock; and `_controller_healthy` accepts **404 as healthy** — any HTTP answer,
even "no such session", proves the ASGI app is up and serving. A subtlety to keep: "the
port answers" is weaker than "*my* process answers" — a stale listener from a previous
run passes this check too, which is why you run `just down` before `just up`.

In [ ]:
for fn in (bringup._wait, bringup._port_open, bringup._controller_healthy):
    print(inspect.getsource(fn))
assert "404" in inspect.getsource(bringup._controller_healthy)
print("→ 404 IS healthy here: an unknown session still proves the app is serving")

And `down()`: read the manifest, `killpg` each process **group** (tolerating
already-dead via `ProcessLookupError`), wait for the sockets to actually release
(SIGTERM is asynchronous), delete the manifest. No manifest → return silently. That is
rule 8's sibling at the ops layer — and the same discipline you saw twice already this
chapter: conftest's finalizer closes the world even on failure, `ChainNetWorld.close`
sweeps the router even on failure, `down` stops the stack even when called twice.
Cleanup, everywhere, assumes it may run against a mess.

In [ ]:
print(inspect.getsource(bringup.down))
down_src = inspect.getsource(bringup.down)
assert "killpg" in down_src and "missing_ok=True" in down_src

**✏️ Your turn 7.1 — prove the no-op**

With no stack running (no manifest), `down()` should be a clean silent no-op. Predict
first — exception, hang, or silence? — then un-comment and run. The scaffold refuses to
touch a *live* stack, so it is safe as-is. Success: `down() -> None` and nothing on your
machine died.

In [ ]:
if bringup.MANIFEST.exists():
    print("skipped: a stack IS running — this exercise wants the empty case (run `just down` first)")
else:
    print("no manifest at", bringup.MANIFEST.name, "— nothing is running")
    # TODO: predict, then un-comment:
    # result = bringup.down()
    # print("down() ->", result, " ← silent no-op: rule 8's ops sibling")
    # assert result is None and not bringup.MANIFEST.exists()

<details><summary>✅ Solution 7.1 — peek only after trying</summary>

```python
result = bringup.down()
assert result is None
```

The first line of `down()` is `if not MANIFEST.exists(): return` — idempotent by
construction. The flip side is worth knowing: delete the manifest while a stack *is*
running and `down()` still no-ops — both processes leak, because the manifest is the
only link between shells.
</details>

## 8. Why profiles beat mocks-everywhere and real-everywhere

Step back and weigh the two designs this chapter quietly rejected.

**Mocks everywhere** is fast and deterministic — and proves only what the mocks claim.
If a fake drifts from the real adapter's behavior, every test passes against the lie.
This repo's defenses are rule 7 (mock and real adapter run the *same* contract-test
suite) and the profile ladder itself: the same lifecycle re-run against reality is a
standing drift alarm.

**Real everywhere** is honest — and needs Foundry, a ~2 GB router lab, and an LLM
endpoint on every run. Nobody pays that per commit; the tests get skipped, and skipped
tests rot into decoration.

The profile ladder takes both halves: CI runs `mock` forever — fast, deterministic,
zero infrastructure — while reality runs locally, on demand, *through the same test
files*. Honesty comes by construction, not willpower: same script, same assertions,
escalating stagings, and a requested reality that cannot silently degrade. The Justfile
makes the default visible:

In [ ]:
justfile = (ROOT / "Justfile").read_text()
recipe = justfile[justfile.index("# run all Python tests") : justfile.index("# ruff lint")]
print(recipe.strip())
assert "uv run pytest" in recipe
print("\n← no SKELETON_PROFILE set, so conftest defaults to mock: CI's reality, forever")

**✏️ Your turn 8.1 — count the multiplication**

How many tests in `test_lifecycle.py` take the `world` fixture? Predict a number, then
count. Each of those runs unchanged on `mock`, `chain`, *and* `chain+net` — N tests × 3
realities, zero test edits. Success: you should see exactly 7 (and be able to say why
the other two need no world).

In [ ]:
# TODO: predict first:  n_prediction = ...

n_all = lifecycle_src.count("def test_")
n_world = lifecycle_src.count("(world):")
print("tests in the file    :", n_all)
print("tests taking `world` :", n_world)
print("lifecycle coverage   :", n_world, "tests × 3 profiles =", n_world * 3, "runs, zero edits")

# un-comment once you've predicted:
# assert n_world == 7

<details><summary>✅ Solution 8.1 — peek only after trying</summary>

```python
assert n_world == 7
```

7 of the 9: the two that need no world are the rule-7 Protocol check
(`test_fakes_satisfy_ports`) and the cross-package proof-template parity test — both are
about *shapes agreeing*, not about any particular reality.
</details>

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| imported `worlds.py`, audited `sys.modules` | the composition root: one place knows what's real; deferred imports keep it cheap | 12 builds its whole stack in one such place |
| read conftest's `world` fixture | `SKELETON_PROFILE` as an environment contract; error-not-skip honesty | the grand finale constructs `ChainWorld`-style wiring directly (12) |
| walked the mock lifecycle by hand | the notebook and `test_lifecycle.py` are twins — CI's forever-play | every later milestone keeps this play green |
| built `ChainWorld` on a live Anvil, same script | THE SWAP: only the binding changed; `isinstance` both readers against the port | 12's real-controller finale |
| read `_ERRORS` / `_fulfill_as` | exception translation at a boundary, with chaining | 13's adversarial matrix attributes attacks to layers the same way |
| read `ChainNetWorld` + its `close()` sweep | template-method hook; zombie policing after failure | lab sessions in 09's recipes |
| toured `up()` / `down()` | process groups, healthchecks, a manifest between shells, idempotent ops | `just up` powers the console and demo |

## Experiments to try

Predict first, then run:

- `SKELETON_PROFILE=banana uv run pytest e2e/tests/test_lifecycle.py` — which of §2's two
  refusals fires, and how many tests even start?
- `SKELETON_PROFILE=chain uv run pytest e2e/tests/test_lifecycle.py -s` — the same
  narration lines as §4, now printed by a real chain run. Time it against the mock run.
- Restart this kernel and rerun §5 **skipping the `seed_chain(anvil)` line** — predict
  both the exception `fulfill` raises and which ticket id a first mint would get, then
  check your reasoning against §5's seeding story.
- The deliberate breakage: run `just up` twice without a `down` in between. Using §7's
  sources, predict what the second `up` does to the first stack's manifest — then clean
  up with `just down` and check `e2e/runs/` for orphans.

## Check yourself

1. Why may only *one* place in a program know which adapter fills which port — what
   breaks when a second place knows?
2. `SKELETON_PROFILE=chain` on a machine without Foundry: skip or error? Why is that the
   right call?
3. In `ChainWorld`, which three things became real — and which organ deliberately stayed
   cardboard?
4. Why does `ChainNetWorld` override only `_make_net`, `provisioned`, `torn_down`, and
   `close`?
5. Your `just up` shell crashed. What file lets a brand-new shell still stop the stack —
   and what leaks if you delete it first?

**Where this goes next:** `12_grand_finale.ipynb` — the full worked example on a real
disposable chain, with the REAL controller (not the stub), real signing, and the
revocation showpiece.

**Deeper dives:** `e2e/notebooks/skeleton_explore.ipynb` (the props, poked further) ·
`e2e/notebooks/chain_client_explore.ipynb` (skeleton v1 as its own tour) ·
`e2e/tests/conftest.py` (the fixture, in situ) · `docs/02-architecture.md` (the import
map this chapter's bindings obey).

In [ ]:
# cleanup: fold the real world away (mock worlds are plain objects — nothing to stop)
if CHAIN_OK:
    cw.close()       # stops the four clients' event-watcher threads
    anvil.stop()
    print("chain world folded away — rerun from the top anytime")
else:
    print("nothing to clean up — the chain section was skipped")